# Deploy MUR to MAAP

Register the four application packages, then run **one** landice job by hand.

Run the cells in order. Everything here is read-only against your own account
except `deploy_algorithm_from_cwl_file` (registers) and `submit_job` (costs
compute).

Prerequisites, from `docs/maap-deployment.html`:

- `./utils/generate_cwl.sh` has run and all four workflows validated
- the four `-dps` images are public on ghcr.io
- the static resources are staged (at least the 8 non-seasonal objects)


## 0. Which Python is this kernel?

This notebook needs only `maap-py`, which the OGC workspace images ship —
nothing from this repo. So the stock kernel is fine and there is nothing to
`pip install` for the cells below.

That changes later: driving `MAAPOrchestrator` needs the repo installed
*into the environment this kernel runs in*. A `pip install -e .` done under a
different interpreter (a venv, or `uv`) will not be visible here. This cell
tells you which is which.


In [ ]:
import sys
print('kernel python :', sys.executable)

try:
    from importlib.metadata import version
    print('maap-py       :', version('maap-py'))
except Exception as e:
    print('maap-py       : NOT INSTALLED --', e)
    print('                this workspace is not on an OGC image')

import importlib.util
have_repo = importlib.util.find_spec('mur_maap') is not None
print('mur_maap      :', 'importable' if have_repo else 'not installed (fine for this notebook)')
if not have_repo:
    print()
    print('  When you need it later, install into THIS interpreter:')
    print(f'    {sys.executable} -m pip install -e ~/mur --no-deps')


## 1. Settings

The only cell you should need to edit. `WORKSPACE_ROOT` comes from
`python utils/upload_static_resources.py --check`.


In [ ]:
from pathlib import Path

VERSION        = '2.0.0'            # must match algorithm_version in the configs
CWL_DIR        = Path.home() / 'mur' / 'maap' / 'cwl_workflows'
WORKSPACE_ROOT = 's3://maap-ops-workspace/jleach_jpl'
STATIC_ROOT    = f'{WORKSPACE_ROOT}/mur/static-resources'

# Ask MAAP ops for the queue names available to you. landice needs only
# 6 GiB, so the smallest queue is fine for the smoke test.
QUEUE = 'maap-dps-worker-8gb'

# The day to smoke-test with.
YEAR, DOY = '2026', '164'

for f in sorted(CWL_DIR.glob('*.cwl')):
    print(f.name)


## 2. Connect

Inside a workspace, authentication is ambient. If this raises, the workspace
is not on an OGC image.


In [ ]:
import json
from maap.maap import MAAP

maap = MAAP()

r = maap.list_algorithms()
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:1500])


## 3. Deploy

Cheapest first. iQUAM has no static inputs and reads no S3, so if it
registers and runs, that proves the app-package shape in isolation.

`202` means *accepted*, not finished — deployment is asynchronous.


In [ ]:
deployments = {}

for m in ['iquam', 'landice', 'l2p', 'mrva']:
    path = CWL_DIR / f'process_mur-{m}_{VERSION}.cwl'
    r = maap.deploy_algorithm_from_cwl_file(file_path=str(path))
    body = r.json() if r.content else {}
    deployments[m] = body.get('deploymentJobID')
    print(f'{m:8} {r.status_code}  {body}')


In [ ]:
# Poll until each deployment settles. Re-run this cell as needed.
for m, dep_id in deployments.items():
    if not dep_id:
        print(f'{m:8} no deployment id -- see the response above')
        continue
    r = maap.get_deployment_status(dep_id)
    status = r.json().get('status') if r.status_code == 200 else r.text
    print(f'{m:8} {status}')


## 4. Find the process ids

Deployment is asynchronous and handed off to an external pipeline — the API
documents a webhook endpoint that third parties call to report status — so a
process does not appear in the catalogue until that reports back. If this
comes back empty, deployment is still in flight; re-run the status cell above.

These cells print the **raw** response. How MAAP identifies a registered
package (bare name, `name:version`, or a numeric id) is not documented, so
read it rather than assume it.


In [ ]:
r = maap.list_algorithms()
print('HTTP', r.status_code)

body = r.json() if r.content else None
print(json.dumps(body, indent=2)[:4000])

# Normalize whatever shape came back into a list, without assuming one.
if isinstance(body, dict):
    algos = body.get('processes') or body.get('algorithms') or []
elif isinstance(body, list):
    algos = body
else:
    algos = []
print()
print(f'{len(algos)} process(es) registered')


In [ ]:
# Every key on the first entry, so the id field names itself.
if algos:
    print(json.dumps(algos[0], indent=2))
else:
    print('Nothing registered yet. Either deployment is still running, or it')
    print('failed -- check the deployment-status cell and the raw call below.')


### Asking the API directly

When a maap-py helper returns something unexpected, go to the endpoint. This
reuses maap-py's own auth header, so it needs no extra credentials.


In [ ]:
import requests

def api(path, **params):
    """GET {maap_api_root}/{path} with maap-py's auth header."""
    url = f"{maap.config.maap_api_root.rstrip('/')}/{path.lstrip('/')}"
    resp = requests.get(url, headers=maap._get_api_header(), params=params or None)
    print(resp.request.url)
    print('HTTP', resp.status_code)
    try:
        return resp.json()
    except Exception:
        print(resp.text[:1000])
        return None

# Every deployment this account has ever started, with its status.
print(json.dumps(api('ogc/deploymentJobs'), indent=2)[:3000])


In [ ]:
# The processes endpoint supports filters the helper may not expose.
print(json.dumps(api('ogc/processes', algorithmName='mur-landice'), indent=2)[:3000])


In [ ]:
# Set this from whatever the cells above show.
LANDICE_PROCESS_ID = 'mur-landice:2.0.0'


## 5. One landice job

**This is the step everything else waits on.** `get_job_result()`'s return
shape is undocumented — a directory prefix, per-file hrefs, or a dict keyed
by CWL output id — and the client code, the output matching and the run-state
machine all depend on which it is.

Run it once, then read the response before building anything against it.


In [ ]:
inputs = {
    'year': YEAR,
    'doy':  DOY,
    'landmask-p01-file':         f'{STATIC_ROOT}/grids/maskGLOBp01deg.gds',
    'gridindex-north-p01-file':  f'{STATIC_ROOT}/mat/p01/saf2north.mat',
    'gridindex-south-p01-file':  f'{STATIC_ROOT}/mat/p01/saf2south.mat',
    'landmask-p011-file':        f'{STATIC_ROOT}/grids/maskGlob1km.gds',
    'gridindex-north-p011-file': f'{STATIC_ROOT}/mat/p011/saf2north.mat',
    'gridindex-south-p011-file': f'{STATIC_ROOT}/mat/p011/saf2south.mat',
}

r = maap.submit_job(
    process_id=LANDICE_PROCESS_ID,
    queue=QUEUE,
    tag='mur.smoke.landice',        # also organises dps_output/
    inputs=inputs,
)
print(r.status_code)
print(json.dumps(r.json(), indent=2))

job_id = r.json().get('id')
job_id


In [ ]:
# Re-run until terminal. landice takes a few minutes.
r = maap.get_job_status(job_id)
print(r.status_code)
print(json.dumps(r.json(), indent=2))


## 6. The response that matters

Print this verbatim and keep it. Everything downstream is written against
whatever shape it turns out to be.


In [ ]:
r = maap.get_job_result(job_id)
print(r.status_code)
print(json.dumps(r.json(), indent=2))


### What to confirm

1. **`output/` holds `p01/` and `p011/` with real files.** Empty means the
   stage-out fix did not reach the image, and nothing downstream can work.
2. **The real filenames** behind `landice_ice_p011`, `landice_grid_p01` and
   `landice_icefiles_p011`.
3. **Which subdirectory `icefiles_YYYY_DDD.txt` lands in** — the file is real
   (written by `landice/src/readosisafice.m`), but its location on DPS is not
   yet established.


In [ ]:
# The outputs also land in the workspace filesystem, so list them directly.
import subprocess
print(subprocess.run(
    ['find', str(Path.home() / 'my-private-bucket' / 'dps_output'),
     '-newermt', '-2 hours', '-type', 'f'],
    capture_output=True, text=True).stdout or '(nothing in the last 2 hours)')


## 7. If something failed

`get_job_metrics` and the job's own logs are the next place to look.


In [ ]:
r = maap.get_job_metrics(job_id)
print(r.status_code)
print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)


In [ ]:
# Every job this session, newest first -- useful after a workspace restart.
r = maap.list_jobs(tag='mur.smoke.landice', limit=10)
print(json.dumps(r.json(), indent=2)[:2000])
